In [ ]:
import re, json, unicodedata
from pathlib import Path  
# ---------------- utilidades ---------------- #
def unaccent(t):
    return ''.join(c for c in unicodedata.normalize("NFD", t)
                   if unicodedata.category(c) != "Mn")

def compact(t):
    return re.sub(r"\s{2,}", " ", t).strip()
  
def merge_hyphen(lines):
    merged, cache = [], None
    for ln in lines:
        if cache:
            merged.append(cache + ln.lstrip()); cache = None; continue
        if ln.rstrip().endswith("-"):
            cache = ln.rstrip()[:-1]
        else:
            merged.append(ln)
    return merged


# ---------------- patrones ------------------ #
RE_CAMARA = re.compile(r'(\d{1,4})/(\d{4})C')
RE_SENADO = re.compile(r'(\d{1,4})/(\d{4})S')
RE_FILA   = re.compile(r'^\s*\d+\s+(\d{1,4})/(\d{4})C')
RE_COMISION = re.compile(r'COMISI[ÓO]N\s+([A-ZÁÉÍÓÚÜÑ ]+)', re.I)


# ----------- tipo de ley flexible ----------- #
def detectar_tipo(tokens):
    i = 0
    while i < len(tokens):
        tk = unaccent(tokens[i]).lower()
        if tk == "acto":
            j, acc = i+1, ""
            while j < len(tokens) and j <= i+4:
                acc += unaccent(tokens[j]).lower()
                if acc.startswith("legisl") and len(acc) >= 9:
                    return "Acto Legislativo", j+1
                j += 1
        elif tk == "ley":
            j, acc = i+1, ""
            while j < len(tokens) and j <= i+4:
                acc += unaccent(tokens[j]).lower()
                if acc.startswith("ordinaria")  and len(acc) >= 9:
                    return "Ley Ordinaria",   j+1
                if acc.startswith("estatutaria") and len(acc) >=10:
                    return "Ley Estatutaria", j+1
                if acc.startswith("organica")   and len(acc) >= 8:
                    return "Ley Orgánica",    j+1
                j += 1
        i += 1
    return None, None


# -------------- parseo de fila -------------- #
def parsear(bloque):
    unido = compact(" ".join(bloque))
    cams = RE_CAMARA.findall(unido)
    if not cams:
        return None

    num_c, anio_c = cams[0]
    num_c_ac, anio_c_ac = (cams[1] if len(cams) > 1 else (None, None))

    sen = RE_SENADO.search(unido)
    num_s, anio_s = sen.groups() if sen else (None, None)

    tokens = unido.split()
    tipo, idx = detectar_tipo(tokens)
    titulo = compact(" ".join(tokens[idx:]).lstrip('“"').rstrip('”"')) if idx else ""

    return {
        "numeroCamara"     : num_c,
        "anioCamara"       : anio_c,
        "numeroCamaraAcum" : num_c_ac,
        "anioCamaraAcum"   : anio_c_ac,
        "numeroSenado"     : num_s,
        "anioSenado"       : anio_s,
        "tipoLey"          : tipo,
        "titulo"           : titulo
    }


# -------- extracción desde TXT ------------ #
def extraer_por_comision(txt_path):
    resultado, bloque, comision = {}, [], None
    with open(txt_path, encoding="utf-8") as f:
        lineas = merge_hyphen(f.read().splitlines())

    for ln in (l.strip() for l in lineas if l.strip()):
        # -------- cabecera ----------
        cab = RE_COMISION.search(ln)
        if cab:
            comision = cab.group(1).strip().upper()
            resultado.setdefault(comision, [])
            continue

        # -------- nueva fila ---------
        if RE_FILA.match(ln):
            # si aún no sabemos la comisión → saltamos la fila
            if comision is None:
                continue
            if bloque:
                meta = parsear(bloque)
                if meta:
                    resultado[comision].append(meta)
                bloque = []
            bloque.append(ln)
        elif bloque:                 # seguimos la misma fila
            bloque.append(ln)

    # última fila
    if bloque and comision:
        meta = parsear(bloque)
        if meta:
            resultado[comision].append(meta)

    return resultado

# ------------------ DEMO ------------------------------------------------- #
if __name__ == "__main__":
    txt_file = r"C:\Users\juans\Documents\pro\Model-Extract-information\document\resource\2020_2021\comison1.txt" 
    comisiones = extraer_por_comision(txt_file)

    # Guarda todo en un único JSON
    Path("proyectos_por_comision.json").write_text(
        json.dumps(comisiones, ensure_ascii=False, indent=4)
    )
    print(json.dumps(comisiones, ensure_ascii=False, indent=4))

{
    "SÉPTIMA": [
        {
            "numeroCamara": "005",
            "anioCamara": "2020",
            "numeroCamaraAcum": null,
            "anioCamaraAcum": null,
            "numeroSenado": null,
            "anioSenado": null,
            "tipoLey": "Ley Ordinaria",
            "titulo": "POR MEDIO DE LA CUAL SE AMPLÍA LA LICENCIA DE MATERNIDAD O PATERNIDAD DURANTE LAS DECLARATORIAS DE EMERGENCIA Y SE DICTAN OTRAS DISPOSICIONES"
        },
        {
            "numeroCamara": "012",
            "anioCamara": "2020",
            "numeroCamaraAcum": null,
            "anioCamaraAcum": null,
            "numeroSenado": null,
            "anioSenado": null,
            "tipoLey": "Ley Ordinaria",
            "titulo": "POR EL CUAL SE ELIMINAN LAS PRÁCTICAS TAURINAS EN EL TERRITORIO NACIONAL Y SE DICTAN OTRAS DISPOSICIONES"
        },
        {
            "numeroCamara": "017",
            "anioCamara": "2020",
            "numeroCamaraAcum": null,
            "anioCamaraAcum":